# Shot-to-Pixel Director

A **local pre-production and generation agent** for a ComfyUI video workflow. You give it a script or an idea. It:

1. **Plans a shot list** the way a cinematographer would (framing, lens, camera move, lighting, duration).
2. **Routes each shot** to the right model: LTX for fast drafts, Wan for VFX elements and control, MiniMax for hero shots.
3. **Writes a prompt per shot** in the style each model responds to.
4. **Queues the generations in ComfyUI**, grouped by model so a 24 GB GPU never has to hold two video models at once.
5. **Lets you review and revise.** You approve or send notes back, and it rewrites the prompt and re-queues.

The language model is a small LLM running in **Foundry Local**, so planning and prompt writing cost nothing and your scripts never leave the machine. Only the video models use the GPU heavily, so the notebook unloads the LLM before generating.

This builds on Lesson 03 (design patterns), Lesson 04 (tool use), Lesson 14 (sequential pipeline) and Lesson 17 (Foundry Local).

> **Dry run by default.** `DRY_RUN = True` means no request is sent to ComfyUI. The notebook writes the patched workflow JSON for each job to disk instead, so you can inspect exactly what would be queued. Set it to `False` once your workflow bindings (section 4) match your ComfyUI graphs.

## Setup

1. Install [Foundry Local](https://learn.microsoft.com/azure/ai-foundry/foundry-local/). This notebook uses the **2.x Python SDK**, which is different from the older `FoundryLocalManager(alias)` API used in Lesson 17.
2. Start ComfyUI as usual (default `http://127.0.0.1:8188`). Not needed while `DRY_RUN = True`.
3. Install the packages:

In [ ]:
%pip install "foundry-local-sdk>=2" httpx -q

## 1. Configuration

In [ ]:
import json
import re
import shutil
import subprocess
import time
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path

import httpx

DRY_RUN = True                       # True = never call ComfyUI; write patched workflows to disk instead
COMFY_URL = "http://127.0.0.1:8188"
LLM_ALIAS = "qwen2.5-7b"             # any chat model alias in the Foundry Local catalog
PROJECT_DIR = Path.cwd() / "shot_to_pixel_demo"
PROJECT_DIR.mkdir(exist_ok=True)

# The overall look of the project. Every prompt inherits this, which keeps shots consistent.
LOOK = (
    "Moody neo-noir, anamorphic feel, shallow depth of field, teal and amber palette, "
    "practical neon light, light film grain, wet reflective streets."
)

## 2. The local LLM (Foundry Local)

`LocalLLM` wraps the Foundry Local 2.x `ChatSession`. A fresh session is created for every call, so calls are independent. The model loads on first use and `unload()` frees its memory. We call `unload()` before any video generation so the 4090's VRAM belongs to ComfyUI.

In [ ]:
class LocalLLM:
    """Small wrapper around a Foundry Local chat model with explicit load/unload."""

    def __init__(self, alias: str):
        self.alias = alias
        self.model = None
        self._loaded = False

    def load(self):
        if self._loaded:
            return
        from foundry_local_sdk import Configuration, FoundryLocalManager

        if FoundryLocalManager.instance is None:
            FoundryLocalManager.initialize(Configuration(app_name="shot_to_pixel"))
        catalog = FoundryLocalManager.instance.catalog
        model = catalog.get_model(self.alias)
        if model is None:
            aliases = sorted({m.alias for m in catalog.list_models()})
            raise ValueError(f"'{self.alias}' is not in the catalog. Available aliases: {aliases}")
        if not model.is_cached:
            model.download(lambda pct: print(f"\rDownloading {self.alias}: {pct:.0f}%", end=""))
            print()
        model.load()
        self.model, self._loaded = model, True

    def unload(self):
        if self._loaded and self.model is not None:
            self.model.unload()
            self._loaded = False
            print(f"Unloaded {self.alias} (VRAM released for ComfyUI).")

    def _generate(self, system: str, user: str, temperature: float, max_tokens: int) -> str:
        from foundry_local_sdk import ChatSession, MessageItem, Request, RequestOptions, SearchOptions, TextItem

        with ChatSession(self.model) as session:
            session.set_options(RequestOptions(search=SearchOptions(
                temperature=temperature, max_output_tokens=max_tokens)))
            with Request().add_item(MessageItem.system(system)).add_item(MessageItem.user(user)) as request:
                with session.process_request(request) as response:
                    return "".join(item.text for item in response if isinstance(item, TextItem))

    def chat(self, system: str, user: str, temperature: float = 0.4, max_tokens: int = 1500) -> str:
        self.load()
        return self._generate(system, user, temperature, max_tokens).strip()


llm = LocalLLM(LLM_ALIAS)
print(f"LLM configured: {LLM_ALIAS} (loads on first use)")

## 3. The shot model and the planner

A `Shot` is the unit of work. The planner asks the LLM for a JSON shot list, then **validates and normalises** every field, because small local models occasionally invent framings or return out-of-range numbers. Nothing downstream trusts raw model output.

Each shot gets an `intent`:

- `standard`: a normal shot, finished on LTX.
- `hero`: a key emotional or visual shot, finished on MiniMax.
- `vfx`: needs a VFX element or tight control, finished on Wan.

In [ ]:
FRAMINGS = ["EWS", "WS", "MS", "MCU", "CU", "ECU"]
CAMERA_MOVES = ["static", "pan", "tilt", "dolly_in", "dolly_out", "track", "handheld", "crane", "orbit"]
INTENTS = ["standard", "hero", "vfx"]


@dataclass
class Shot:
    id: str
    beat: str
    framing: str = "MS"
    lens_mm: int = 35
    camera: str = "static"
    lighting: str = ""
    duration_s: float = 4.0
    intent: str = "standard"
    model: str = ""                                  # final-pass model, set by route()
    seed: int = 0
    status: str = "planned"                          # planned -> drafted -> approved -> finished
    notes: str = ""                                  # your director's notes from revise()
    prompts: dict = field(default_factory=dict)      # model -> prompt text
    outputs: dict = field(default_factory=dict)      # stage -> list of file paths


def _pick(value, allowed, default):
    value = str(value).strip().lower().replace(" ", "_").replace("-", "_")
    match = [a for a in allowed if a.lower() == value]
    return match[0] if match else default


def normalise_shot(raw: dict, index: int) -> Shot:
    """Coerce one model-produced dict into a valid Shot."""
    try:
        lens = int(float(raw.get("lens_mm", 35)))
    except (TypeError, ValueError):
        lens = 35
    try:
        duration = float(raw.get("duration_s", 4))
    except (TypeError, ValueError):
        duration = 4.0
    return Shot(
        id=f"S{index:02d}",
        beat=str(raw.get("beat", "")).strip() or f"Shot {index}",
        framing=_pick(raw.get("framing", "MS"), FRAMINGS, "MS"),
        lens_mm=min(max(lens, 14), 200),
        camera=_pick(raw.get("camera", "static"), CAMERA_MOVES, "static"),
        lighting=str(raw.get("lighting", "")).strip(),
        duration_s=min(max(duration, 2.0), 8.0),
        intent=_pick(raw.get("intent", "standard"), INTENTS, "standard"),
        seed=1000 + index * 7,
    )


def extract_json_array(text: str) -> list:
    """Pull the first JSON array out of a model reply, tolerating code fences and chatter."""
    text = re.sub(r"```(?:json)?", "", text)
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end <= start:
        raise ValueError("no JSON array found")
    data = json.loads(text[start:end + 1])
    if not isinstance(data, list) or not all(isinstance(x, dict) for x in data):
        raise ValueError("expected a list of objects")
    return data


def plan_shots(script: str, max_shots: int = 8, attempts: int = 3) -> list[Shot]:
    system = (
        "You are a cinematographer planning a shot list. Reply with ONLY a JSON array and no other text. "
        "Each item must be an object with these keys: "
        f'"beat" (one sentence describing what happens), "framing" (one of {FRAMINGS}), '
        '"lens_mm" (integer 14-200), '
        f'"camera" (one of {CAMERA_MOVES}), "lighting" (short phrase), "duration_s" (number 2-8), '
        f'"intent" (one of {INTENTS}). '
        'Use intent "hero" for the one or two most important shots, "vfx" for shots that need a visual '
        'effect element, and "standard" for everything else. '
        f"Use at most {max_shots} shots and cover the script in order."
    )
    user = f"Project look: {LOOK}\n\nScript:\n{script}"
    last_error = None
    for attempt in range(attempts):
        reply = llm.chat(system, user, temperature=0.3 if attempt == 0 else 0.1)
        try:
            raw = extract_json_array(reply)
            if not raw:
                raise ValueError("empty shot list")
            return [normalise_shot(item, i + 1) for i, item in enumerate(raw[:max_shots])]
        except (ValueError, json.JSONDecodeError) as exc:
            last_error = exc
            user += "\n\nYour last reply was not valid. Reply with ONLY the JSON array."
    raise RuntimeError(f"Could not get a valid shot list after {attempts} attempts: {last_error}")


def board(shots: list[Shot]) -> None:
    """Print the current state of every shot."""
    print(f"{'ID':<4} {'FRAME':<5} {'LENS':<6} {'CAMERA':<9} {'SEC':<4} {'INTENT':<9} {'MODEL':<8} STATUS")
    for s in shots:
        print(f"{s.id:<4} {s.framing:<5} {str(s.lens_mm) + 'mm':<6} {s.camera:<9} {s.duration_s:<4.0f} "
              f"{s.intent:<9} {s.model or '-':<8} {s.status}")
        print(f"     {s.beat}")

## 4. Model profiles, router and workflow bindings

### Router

Routing is **plain Python, not the LLM**. A 7B model asked to pick between three video models will be inconsistent, and the rule is simple enough to write down. Edit `ROUTES` if you disagree.

Every shot also gets a quick **LTX draft** first. You judge framing and motion on the cheap model, approve, and only then spend the slow models on finals.

### Profiles

Each profile holds what is true about that model's output: frame rate, the frame-count rule, and default resolution. LTX needs frame counts of the form `8n+1` and Wan `4n+1`, so `frames_for()` snaps durations to a legal count. The `style` text is what the LLM follows when writing prompts for that model.

> **Check these against your setup.** Resolutions and frame rates are reasonable defaults, not measured on your machine. The MiniMax profile in particular is a guess because I have not seen your local build. Edit it to match what it responds to.

### Workflow bindings

ComfyUI runs a workflow when you POST its **API-format** JSON. To get yours: in ComfyUI enable *Settings → Dev mode*, then use **Save (API Format)**. Put the files in `workflows/` and fill in `bind` below. Each entry maps a field to `(node_id, input_name)` in your graph. You can read the node ids in the exported JSON. The ids below are placeholders.

In [ ]:
MODEL_PROFILES = {
    "ltx": {
        "fps": 24, "frame_step": 8, "width": 768, "height": 512,
        "draft_width": 512, "draft_height": 320,
        "style": (
            "One flowing paragraph of 3 to 5 sentences in present tense. Describe the action in the order it "
            "happens. Include the camera movement, the lighting and the atmosphere. Be concrete and visual."
        ),
    },
    "wan": {
        "fps": 16, "frame_step": 4, "width": 832, "height": 480,
        "style": (
            "Order the prompt as: subject and appearance, setting, motion, lighting and colour, then camera "
            "and shot size. Keep one clear action. Use precise cinematic terms."
        ),
    },
    "minimax": {
        "fps": 24, "frame_step": 1, "width": 1280, "height": 720,
        "style": (
            "Two or three vivid sentences. Describe the subject, the motion and the mood, and state the camera "
            "movement explicitly in plain words."
        ),
    },
}

NEGATIVE = "blurry, low quality, jitter, flicker, warped faces, extra limbs, text, watermark, logo"

# intent -> final-pass model
ROUTES = {"standard": "ltx", "hero": "minimax", "vfx": "wan"}
DRAFT_MODEL = "ltx"
MODEL_ORDER = ["ltx", "wan", "minimax"]   # batch order, so each model loads once per pass

# EDIT ME: replace files and node ids with the ones from your exported API-format workflows.
WORKFLOWS = {
    "ltx": {"file": "workflows/ltx_api.json", "bind": {
        "prompt": ("6", "text"), "negative": ("7", "text"), "seed": ("3", "noise_seed"),
        "width": ("20", "width"), "height": ("20", "height"), "frames": ("20", "length"),
        "fps": ("22", "frame_rate"), "prefix": ("30", "filename_prefix")}},
    "wan": {"file": "workflows/wan_api.json", "bind": {
        "prompt": ("6", "text"), "negative": ("7", "text"), "seed": ("3", "seed"),
        "width": ("20", "width"), "height": ("20", "height"), "frames": ("20", "length"),
        "fps": ("22", "fps"), "prefix": ("30", "filename_prefix")}},
    "minimax": {"file": "workflows/minimax_api.json", "bind": {
        "prompt": ("6", "text"), "negative": ("7", "text"), "seed": ("3", "seed"),
        "width": ("20", "width"), "height": ("20", "height"), "frames": ("20", "length"),
        "fps": ("22", "fps"), "prefix": ("30", "filename_prefix")}},
}


def route(shots: list[Shot]) -> None:
    for s in shots:
        s.model = ROUTES[s.intent]


def frames_for(duration_s: float, model: str) -> int:
    """Snap a duration to a frame count the model accepts (8n+1 for LTX, 4n+1 for Wan)."""
    p = MODEL_PROFILES[model]
    raw = max(1, round(duration_s * p["fps"]))
    step = p["frame_step"]
    return round((raw - 1) / step) * step + 1 if step > 1 else raw

## 5. Writing prompts

The LLM rewrites each shot into a prompt in the target model's style, using the project `LOOK` and any notes you gave. If the model fails or returns nothing, a template built from the shot fields is used instead, so the pipeline never stops on a prompt.

In [ ]:
FRAMING_WORDS = {
    "EWS": "extreme wide shot", "WS": "wide shot", "MS": "medium shot",
    "MCU": "medium close-up", "CU": "close-up", "ECU": "extreme close-up",
}


def fallback_prompt(shot: Shot) -> str:
    camera = shot.camera.replace("_", " ")
    return (f"{shot.beat} {FRAMING_WORDS[shot.framing].capitalize()}, {shot.lens_mm}mm lens, {camera} camera. "
            f"{shot.lighting}. {LOOK}").replace("  ", " ")


def write_prompt(shot: Shot, model: str) -> str:
    system = (
        f"You write text-to-video prompts for the {model.upper()} model. {MODEL_PROFILES[model]['style']} "
        "Reply with ONLY the prompt text, no quotes, no labels, no commentary."
    )
    user = (
        f"Beat: {shot.beat}\n"
        f"Shot size: {FRAMING_WORDS[shot.framing]}, {shot.lens_mm}mm lens\n"
        f"Camera: {shot.camera.replace('_', ' ')}\n"
        f"Lighting: {shot.lighting or 'motivated, cinematic'}\n"
        f"Duration: {shot.duration_s:.0f} seconds\n"
        f"Project look: {LOOK}\n"
        + (f"Director's notes (follow these): {shot.notes}\n" if shot.notes else "")
    )
    try:
        text = llm.chat(system, user, temperature=0.6, max_tokens=300).strip().strip('"')
    except Exception as exc:  # a prompt failure must not stop the batch
        print(f"  prompt writer failed for {shot.id} ({exc}); using the template")
        text = ""
    return text or fallback_prompt(shot)


def prepare_prompts(jobs: list["Job"], shots: dict[str, Shot]) -> None:
    """Write any missing prompts. Do this BEFORE run_jobs(), which unloads the LLM."""
    for job in jobs:
        shot = shots[job.shot_id]
        if job.model not in shot.prompts:
            shot.prompts[job.model] = write_prompt(shot, job.model)

## 6. ComfyUI client

Four small functions cover the API: queue a workflow, wait for it, download the result, and free VRAM.

- `POST /prompt` queues a workflow and returns a `prompt_id`.
- `GET /history/{prompt_id}` shows when it is done and lists its output files.
- `GET /view` downloads an output file.
- `POST /free` unloads models and frees memory. We call it **when switching from one video model to the next**.

`patch_workflow()` writes the shot's values into your exported workflow using the `bind` table. It fails loudly if a node id is missing, so a wrong binding cannot silently produce the wrong video.

In [ ]:
def load_workflow(model: str) -> tuple[dict, dict]:
    spec = WORKFLOWS[model]
    path = Path(spec["file"])
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")), spec["bind"]
    if DRY_RUN:
        # No exported workflow yet: build a stub holding exactly the bound inputs so the demo runs.
        stub = {}
        for node, name in spec["bind"].values():
            stub.setdefault(node, {"class_type": "STUB", "inputs": {}})["inputs"][name] = None
        return stub, spec["bind"]
    raise FileNotFoundError(
        f"{path} not found. Export your {model} graph with 'Save (API Format)' and update WORKFLOWS."
    )


def patch_workflow(workflow: dict, bind: dict, values: dict) -> dict:
    wf = json.loads(json.dumps(workflow))   # deep copy
    for key, value in values.items():
        if key not in bind:
            continue
        node, name = bind[key]
        if node not in wf:
            raise KeyError(f"Binding '{key}' points at node {node}, which is not in the workflow.")
        wf[node].setdefault("inputs", {})[name] = value
    return wf


def comfy_queue(workflow: dict, client_id: str) -> str:
    r = httpx.post(f"{COMFY_URL}/prompt", json={"prompt": workflow, "client_id": client_id}, timeout=30)
    r.raise_for_status()
    return r.json()["prompt_id"]


def _collect_files(outputs: dict) -> list[dict]:
    """Gather every real output file, whatever the save node calls its output key."""
    files = []
    for node_output in outputs.values():
        for value in node_output.values():
            if isinstance(value, list):
                files += [f for f in value if isinstance(f, dict) and "filename" in f
                          and f.get("type", "output") == "output"]
    return files


def comfy_wait(prompt_id: str, timeout_s: int = 3600, poll_s: float = 3.0) -> list[dict]:
    start = time.time()
    while time.time() - start < timeout_s:
        history = httpx.get(f"{COMFY_URL}/history/{prompt_id}", timeout=30).json()
        if prompt_id in history:
            entry = history[prompt_id]
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError(f"ComfyUI reported an error for {prompt_id}: {entry['status']}")
            return _collect_files(entry.get("outputs", {}))
        time.sleep(poll_s)
    raise TimeoutError(f"{prompt_id} did not finish within {timeout_s}s")


def comfy_download(file: dict, dest_dir: Path, stem: str) -> Path:
    r = httpx.get(f"{COMFY_URL}/view", params={
        "filename": file["filename"], "subfolder": file.get("subfolder", ""), "type": file.get("type", "output")},
        timeout=300)
    r.raise_for_status()
    dest = dest_dir / f"{stem}{Path(file['filename']).suffix}"
    dest.write_bytes(r.content)
    return dest


def comfy_free() -> None:
    httpx.post(f"{COMFY_URL}/free", json={"unload_models": True, "free_memory": True}, timeout=60)

## 7. Jobs and the batch runner

A `Job` is one generation: a shot, a stage (`draft` or `final`) and a model. The runner:

1. **Unloads the LLM** so ComfyUI has the whole 24 GB.
2. **Sorts jobs by model** (`MODEL_ORDER`) so each video model loads once.
3. Calls `comfy_free()` **between models**.
4. Runs jobs **one at a time**. Queueing several at once would not make a single GPU faster.
5. Saves outputs under `renders/<stage>/` and records them on the shot.

Drafts use the LTX profile's smaller `draft_width`/`draft_height` and cap the clip at 3 seconds. That is enough to judge framing and motion.

In [ ]:
@dataclass
class Job:
    shot_id: str
    stage: str      # "draft" or "final"
    model: str


def build_jobs(shots: list[Shot], stage: str) -> list[Job]:
    if stage == "draft":
        return [Job(s.id, "draft", DRAFT_MODEL) for s in shots if s.status == "planned"]
    return [Job(s.id, "final", s.model) for s in shots if s.status == "approved"]


def job_values(shot: Shot, job: Job) -> dict:
    p = MODEL_PROFILES[job.model]
    draft = job.stage == "draft"
    duration = min(shot.duration_s, 3.0) if draft else shot.duration_s
    return {
        "prompt": shot.prompts[job.model],
        "negative": NEGATIVE,
        "seed": shot.seed,
        "width": p.get("draft_width", p["width"]) if draft else p["width"],
        "height": p.get("draft_height", p["height"]) if draft else p["height"],
        "frames": frames_for(duration, job.model),
        "fps": p["fps"],
        "prefix": f"{shot.id}_{job.stage}_{job.model}",
    }


def run_jobs(jobs: list[Job], shots: dict[str, Shot]) -> None:
    llm.unload()   # give the GPU to the video models
    jobs = sorted(jobs, key=lambda j: (MODEL_ORDER.index(j.model), j.shot_id))
    out_root = PROJECT_DIR / ("dry_run" if DRY_RUN else "renders")
    client_id, previous_model = str(uuid.uuid4()), None

    for job in jobs:
        shot = shots[job.shot_id]
        if previous_model and job.model != previous_model and not DRY_RUN:
            comfy_free()
        previous_model = job.model

        workflow, bind = load_workflow(job.model)
        values = job_values(shot, job)
        patched = patch_workflow(workflow, bind, values)
        stem = values["prefix"]
        dest_dir = out_root / job.stage
        dest_dir.mkdir(parents=True, exist_ok=True)

        if DRY_RUN:
            path = dest_dir / f"{stem}.workflow.json"
            path.write_text(json.dumps(patched, indent=2), encoding="utf-8")
            saved = [str(path)]
        else:
            print(f"{job.shot_id} {job.stage} on {job.model}: queueing ({values['frames']} frames)...")
            files = comfy_wait(comfy_queue(patched, client_id))
            saved = [str(comfy_download(f, dest_dir, f"{stem}_{i}")) for i, f in enumerate(files)]

        shot.outputs[job.stage] = saved
        shot.status = "drafted" if job.stage == "draft" else "finished"
        print(f"  {job.shot_id} {job.stage:<5} {job.model:<8} {values['width']}x{values['height']} "
              f"{values['frames']}f -> {Path(saved[0]).name if saved else '(no output)'}")

## 8. Review: checks, approval and revision

The agent can't *see* the video. Foundry Local's text models can't judge an image, so I do not pretend they can. Review is split honestly:

- **Automatic checks** with `ffprobe` (if installed): the file exists, and its resolution and duration match what was asked.
- **You** look at the clips and decide. `approve()` sends shots to the final pass. `revise()` takes your note in plain language (for example *"too shaky, slower push-in, less haze"*), stores it as a director's note, bumps the seed, and sends the shot back to be re-drafted with a rewritten prompt.

In [ ]:
def probe(path: str) -> dict | None:
    """Return width, height and duration for a video file using ffprobe, or None if unavailable."""
    if not shutil.which("ffprobe") or Path(path).suffix.lower() not in {".mp4", ".webm", ".mov", ".mkv"}:
        return None
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
         "stream=width,height:format=duration", "-of", "json", path],
        capture_output=True, text=True)
    if out.returncode != 0:
        return None
    info = json.loads(out.stdout)
    stream = info["streams"][0]
    return {"width": stream["width"], "height": stream["height"], "duration": float(info["format"]["duration"])}


def check_outputs(shots: list[Shot], stage: str) -> None:
    for s in shots:
        for path in s.outputs.get(stage, []):
            info = probe(path)
            if info is None:
                continue
            expected = min(s.duration_s, 3.0) if stage == "draft" else s.duration_s
            flag = "" if abs(info["duration"] - expected) <= 0.75 else f"  <-- expected about {expected:.1f}s"
            print(f"{s.id} {Path(path).name}: {info['width']}x{info['height']}, {info['duration']:.1f}s{flag}")


def approve(shots: dict[str, Shot], *ids: str) -> None:
    for i in ids:
        if shots[i].status != "drafted":
            print(f"{i}: only drafted shots can be approved (status is {shots[i].status})")
            continue
        shots[i].status = "approved"


def revise(shots: dict[str, Shot], shot_id: str, feedback: str) -> None:
    shot = shots[shot_id]
    shot.notes = (shot.notes + " " + feedback).strip()
    shot.seed += 1
    shot.prompts.clear()
    shot.outputs.clear()
    shot.status = "planned"
    print(f"{shot_id}: note saved, seed -> {shot.seed}, back to 'planned' for a new draft.")

## 9. Export

`export_project()` writes `shots.json` (the full state, so you can reload a project) and `shotlist.md`, a readable shot list with the prompt used for each model. The Markdown file is useful on its own as a pre-production document.

In [ ]:
def export_project(shots: list[Shot]) -> None:
    (PROJECT_DIR / "shots.json").write_text(json.dumps([asdict(s) for s in shots], indent=2), encoding="utf-8")
    lines = ["# Shot list", "", f"**Look:** {LOOK}", "",
             "| ID | Frame | Lens | Camera | Sec | Intent | Model | Status |",
             "|---|---|---|---|---|---|---|---|"]
    for s in shots:
        lines.append(f"| {s.id} | {s.framing} | {s.lens_mm}mm | {s.camera} | {s.duration_s:.0f} | "
                     f"{s.intent} | {s.model} | {s.status} |")
    for s in shots:
        lines += ["", f"## {s.id}: {s.beat}", f"Lighting: {s.lighting or 'n/a'}"]
        if s.notes:
            lines.append(f"Director's notes: {s.notes}")
        for model, prompt in s.prompts.items():
            lines += [f"- **{model}** (seed {s.seed}): {prompt}"]
    (PROJECT_DIR / "shotlist.md").write_text("\n".join(lines), encoding="utf-8")
    print(f"Wrote {PROJECT_DIR / 'shots.json'} and {PROJECT_DIR / 'shotlist.md'}")

## 10. Run it

### Step 1: plan the shots

Replace `SCRIPT` with your own scene or idea. The first call downloads the model (once) and loads it.

In [ ]:
SCRIPT = """
Night, a rain-soaked alley behind a closed noodle shop. A courier, MARA, stops under a flickering
neon sign and checks a small glowing parcel. Behind her, a footstep echoes. She turns: nobody there.
She looks back at the parcel and its glow has gone cold. The neon sign dies and the alley goes black.
"""

shot_list = plan_shots(SCRIPT)
route(shot_list)
shots = {s.id: s for s in shot_list}
board(shot_list)

### Step 2: draft everything on LTX

This writes LTX prompts, then unloads the LLM and queues the drafts. With `DRY_RUN = True`, you will see the files that *would* be queued under `dry_run/draft/`.

In [ ]:
draft_jobs = build_jobs(shot_list, "draft")
prepare_prompts(draft_jobs, shots)
run_jobs(draft_jobs, shots)
check_outputs(shot_list, "draft")
board(shot_list)

### Step 3: review, revise, approve

Watch the drafts, then tell the agent what you think. The example below revises one shot and approves the rest. Change it to match your own judgement.

In [ ]:
first = shot_list[0].id
revise(shots, first, "slower push-in, less haze, keep the neon reflections on the wet ground")

# The revised shot goes back through the draft pass with its new note, seed and prompt.
redo = build_jobs(shot_list, "draft")
prepare_prompts(redo, shots)
run_jobs(redo, shots)

approve(shots, *[s.id for s in shot_list])
board(shot_list)

### Step 4: finals on the right model for each shot

Approved shots are generated on their routed model, batched by model. The LLM writes the missing final prompts first and is unloaded just before generation starts.

In [ ]:
final_jobs = build_jobs(shot_list, "final")
prepare_prompts(final_jobs, shots)
run_jobs(final_jobs, shots)
board(shot_list)
export_project(shot_list)

## Going live

1. Export your three ComfyUI graphs with **Save (API Format)** into a `workflows/` folder next to this notebook.
2. Open each JSON and update the node ids in `WORKFLOWS[...]["bind"]` (section 4). A text node's id is the key; its input name is usually `text`.
3. Run one model first with a single short shot. Confirm the clip matches the prompt, width, height and frame count.
4. Set `DRY_RUN = False`.

## Known limits

- **It cannot watch your clips.** Review is `ffprobe` checks plus your own judgement.
- **Prompt styles are starting points.** Tune the `style` text per model as you learn what each one responds to.
- **Resolutions and frame rates are defaults**, not measured on your machine. LTX and Wan frame rules (`8n+1`, `4n+1`) are real. MiniMax is unverified.
- **Not covered:** characters staying consistent across shots (that needs reference images or LoRAs in your graphs), audio, and stitching the final edit.

## Challenges

1. Add a **continuity** field so the planner repeats character and prop descriptions in every prompt.
2. Add a **reference image** binding so an approved still can drive the final pass (image-to-video).
3. Replace the human review step with a **local vision model** in Foundry Local that scores a draft's contact sheet.
4. Add a `Montage` step that writes an EDL or a DaVinci Resolve import list from `shots.json`.